In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, rank, dense_rank
data = [
(1, "Ravi Kumar", "Engineering", "Hyderabad", 92000),
(2, "Sneha Reddy", "Engineering", "Bengaluru", 97000),
(3, "Amit Shah", "Finance", "Mumbai", 88000),
(4, "Pooja Nair", "HR", "Chennai", 69000),
(5, "Nikhil Verma", "Engineering", "Delhi", 85000),
(6, "Meera Iyer", "Finance", "Hyderabad", 91000),
(7, "Karan Malhotra", "Marketing", "Mumbai", 76000),
(8, "Anjali Rao", "HR", "Bengaluru", 72000),
(9, "Vivek Joshi", "Finance", "Delhi", 94000),
(10, "Fatima Khan", "Marketing", "Chennai", 81000),
(11, "Aditya Menon", "Engineering", "Mumbai", 99000),
(12, "Lakshmi Das", "Finance", "Bengaluru", 87000)
]
columns = ["emp_id", "emp_name", "department", "city", "salary"]
df = spark.createDataFrame(data, columns)
display(df)

emp_id,emp_name,department,city,salary
1,Ravi Kumar,Engineering,Hyderabad,92000
2,Sneha Reddy,Engineering,Bengaluru,97000
3,Amit Shah,Finance,Mumbai,88000
4,Pooja Nair,HR,Chennai,69000
5,Nikhil Verma,Engineering,Delhi,85000
6,Meera Iyer,Finance,Hyderabad,91000
7,Karan Malhotra,Marketing,Mumbai,76000
8,Anjali Rao,HR,Bengaluru,72000
9,Vivek Joshi,Finance,Delhi,94000
10,Fatima Khan,Marketing,Chennai,81000


In [0]:
windowspec = Window.partitionBy("department").orderBy(col("salary").desc())
df = df.withColumn("rank", rank().over(windowspec))
df.show()

+------+--------------+-----------+---------+------+----+
|emp_id|      emp_name| department|     city|salary|rank|
+------+--------------+-----------+---------+------+----+
|    11|  Aditya Menon|Engineering|   Mumbai| 99000|   1|
|     2|   Sneha Reddy|Engineering|Bengaluru| 97000|   2|
|     1|    Ravi Kumar|Engineering|Hyderabad| 92000|   3|
|     5|  Nikhil Verma|Engineering|    Delhi| 85000|   4|
|     9|   Vivek Joshi|    Finance|    Delhi| 94000|   1|
|     6|    Meera Iyer|    Finance|Hyderabad| 91000|   2|
|     3|     Amit Shah|    Finance|   Mumbai| 88000|   3|
|    12|   Lakshmi Das|    Finance|Bengaluru| 87000|   4|
|     8|    Anjali Rao|         HR|Bengaluru| 72000|   1|
|     4|    Pooja Nair|         HR|  Chennai| 69000|   2|
|    10|   Fatima Khan|  Marketing|  Chennai| 81000|   1|
|     7|Karan Malhotra|  Marketing|   Mumbai| 76000|   2|
+------+--------------+-----------+---------+------+----+



In [0]:
from pyspark.sql.functions import when, col
df = df.withColumn("salary_level", when(col("salary") >= 95000, "very high").when(col("salary") >= 90000, "high").when(col("salary") >= 85000, "medium").when(col("salary") >= 80000, "low").otherwise("very low"))

df.show()

+------+--------------+-----------+---------+------+----+------------+
|emp_id|      emp_name| department|     city|salary|rank|salary_level|
+------+--------------+-----------+---------+------+----+------------+
|    11|  Aditya Menon|Engineering|   Mumbai| 99000|   1|   very high|
|     2|   Sneha Reddy|Engineering|Bengaluru| 97000|   2|   very high|
|     1|    Ravi Kumar|Engineering|Hyderabad| 92000|   3|        high|
|     5|  Nikhil Verma|Engineering|    Delhi| 85000|   4|      medium|
|     9|   Vivek Joshi|    Finance|    Delhi| 94000|   1|        high|
|     6|    Meera Iyer|    Finance|Hyderabad| 91000|   2|        high|
|     3|     Amit Shah|    Finance|   Mumbai| 88000|   3|      medium|
|    12|   Lakshmi Das|    Finance|Bengaluru| 87000|   4|      medium|
|     8|    Anjali Rao|         HR|Bengaluru| 72000|   1|    very low|
|     4|    Pooja Nair|         HR|  Chennai| 69000|   2|    very low|
|    10|   Fatima Khan|  Marketing|  Chennai| 81000|   1|         low|
|     

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
windows = Window.partitionBy("department").orderBy(col("salary").desc())

top_two = df.withColumn("rn", row_number().over(windows)).filter(col("rn") <= 2)
top_two.show()

+------+--------------+-----------+---------+------+----+------------+---+
|emp_id|      emp_name| department|     city|salary|rank|salary_level| rn|
+------+--------------+-----------+---------+------+----+------------+---+
|    11|  Aditya Menon|Engineering|   Mumbai| 99000|   1|   very high|  1|
|     2|   Sneha Reddy|Engineering|Bengaluru| 97000|   2|   very high|  2|
|     9|   Vivek Joshi|    Finance|    Delhi| 94000|   1|        high|  1|
|     6|    Meera Iyer|    Finance|Hyderabad| 91000|   2|        high|  2|
|     8|    Anjali Rao|         HR|Bengaluru| 72000|   1|    very low|  1|
|     4|    Pooja Nair|         HR|  Chennai| 69000|   2|    very low|  2|
|    10|   Fatima Khan|  Marketing|  Chennai| 81000|   1|         low|  1|
|     7|Karan Malhotra|  Marketing|   Mumbai| 76000|   2|    very low|  2|
+------+--------------+-----------+---------+------+----+------------+---+



In [0]:
from pyspark.sql.functions import avg

avg_salary_df = df.groupBy("department") \
                  .agg(avg("salary").alias("avg_salary"))

display(avg_salary_df)

department,avg_salary
Engineering,93250.0
Finance,90000.0
HR,70500.0
Marketing,78500.0
